In [1]:
# Starting from the ORIGINAL dataset so we can engineer from
# un-scaled Time and Amount columns, then re-apply preprocessing

import pandas as pd
import numpy as np

# Load the original dataset
df = pd.read_csv("../data/creditcard.csv")
print("Loaded dataset shape:", df.shape)
df[["Time", "Amount"]].head()

Loaded dataset shape: (284807, 31)


,Time,Amount
0,0.0,149.62
1,0.0,2.69
2,1.0,378.66
3,1.0,123.50
4,2.0,69.99


In [2]:
# Make a copy to work on
df_fe = df.copy()

# Feature 1: Hour of day
# 'Time' is seconds elapsed since the first transaction. The data spans 2 days,
# so we convert seconds to an hour of day value (0-23) to capture daily pattern.
df_fe["Hour"] = (df_fe["Time"] // 3600) % 24

# Feature 2: Log transformed amount
# Transaction amounts are very skewed (many small, a few very large).
# A log transform compress that range so models handle it better.
# We use log1p (log of 1 + amount) so that an amount of 0 doesn't cause an error.
df_fe["Log_Amount"] = np.log1p(df_fe["Amount"])

# Show the new features next to the originals
df_fe[["Time", "Hour", "Amount", "Log_Amount"]].head(10)

,Time,Hour,Amount,Log_Amount
0,0.0,0.0,149.62,5.014760
1,0.0,0.0,2.69,1.305626
2,1.0,0.0,378.66,5.939276
3,1.0,0.0,123.50,4.824306
4,2.0,0.0,69.99,4.262539
5,2.0,0.0,3.67,1.541159
6,4.0,0.0,4.99,1.790091
7,7.0,0.0,40.80,3.732896
8,7.0,0.0,93.20,4.545420
9,9.0,0.0,3.68,1.543298


In [3]:
# Confirm the Hour feature actually spans the full 0-23 range across the dataset
print("Unique hours present:", sorted(df_fe["Hour"].unique()))
print("Min hour:", df_fe["Hour"].min(), "| Max hour:", df_fe["Hour"].max())

Unique hours present: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0)]
Min hour: 0.0 | Max hour: 23.0


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import os

# Drop the original 'Time' column - we now have 'Hour' instead.
# keep 'Amount' and 'Log_Amount' - the models can use both.
df_model = df_fe.drop("Time", axis=1)

# Scale the non-PCA numeric columns so they match the scale of V1-V28
scaler = StandardScaler()
cols_to_scale = ["Amount", "Log_Amount", "Hour"]
df_model[cols_to_scale] = scaler.fit_transform(df_model[cols_to_scale])

# Separate features (X) from target (y)
X = df_model.drop("Class", axis=1)
y = df_model["Class"]

# Stratified 80/20 train-test split (same as preprocessing notebook).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Apply SMOTE to the training set ONLY
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Save this feature engineered version to a separate subfolder
os.makedirs("../data/processed_fe", exist_ok=True)
X_train_balanced.to_csv("../data/processed_fe/X_train_balanced.csv", index=False)
y_train_balanced.to_csv("../data/processed_fe/y_train_balanced.csv", index=False)
X_test.to_csv("../data/processed_fe/X_test.csv", index=False)
y_test.to_csv("../data/processed_fe/y_test.csv", index=False)

print("Feature-engineered data saved to data/processed_fe/")
print("Training features now include:", list(X.columns))

Feature-engineered data saved to data/processed_fe/
Training features now include: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour', 'Log_Amount']
